# Lab 9: Open-Source Models with Hugging Face – Audio Tasks


In [1]:
!pip install transformers torch torchaudio datasets soundfile librosa

## Task 1: Zero-Shot Audio Classification


In [1]:
from transformers import pipeline
import requests

# Load zero-shot audio classification pipeline
classifier = pipeline(
    task="zero-shot-audio-classification",
    model="laion/clap-htsat-unfused"
)

# Download a sample audio file
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
audio_path = "sample_audio.flac"

with open(audio_path, "wb") as f:
    f.write(requests.get(audio_url).content)

# Define candidate labels
candidate_labels = ["speech", "music", "noise", "applause", "silence"]

# Classify the audio
results = classifier(audio_path, candidate_labels=candidate_labels)
print("Zero-Shot Audio Classification Results:")
for result in results:
    print(f"  {result['label']}: {result['score']:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/615M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/614M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Zero-Shot Audio Classification Results:
  speech: 0.7657
  music: 0.1319
  noise: 0.0713
  applause: 0.0222
  silence: 0.0089


## Task 2: Automatic Speech Recognition (ASR) with Whisper


In [3]:
from transformers import pipeline

asr = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-base",
    chunk_length_s=30
)

transcription = asr(audio_path, return_timestamps=False)
print("Transcription:")
print(transcription["text"])


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
A 

Transcription:
 I have a dream that one day this nation will rise up and live out the true meaning of its creed.


In [4]:
# ASR with timestamps
transcription_with_timestamps = asr(
    audio_path,
    return_timestamps=True
)
print("Transcription with Timestamps:")
for chunk in transcription_with_timestamps["chunks"]:
    start, end = chunk["timestamp"]
    print(f"  [{start:.1f}s - {end:.1f}s]: {chunk['text']}")

Transcription with Timestamps:
  [0.0s - 11.7s]:  I have a dream that one day this nation will rise up and live out the true meaning of
  [11.7s - 12.4s]:  its creed.


## Task 3: Text-to-Speech (TTS)


In [6]:
import soundfile as sf
from transformers import pipeline

tts = pipeline("text-to-speech", model="facebook/mms-tts-eng")

text = "Hello! Welcome to the Foundations of Generative AI lab. Today we are learning about audio models."
speech = tts(text)

output_path = "generated_speech.wav"
sf.write(output_path, speech["audio"], speech["sampling_rate"])
print(f"Generated speech saved to: {output_path}")
print(f"Sampling rate: {speech['sampling_rate']} Hz")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

Generated speech saved to: generated_speech.wav
Sampling rate: 16000 Hz


In [7]:
# Play the audio in notebook (works in Jupyter)
from IPython.display import Audio
Audio(output_path)

## Task 4: Language Identification from Speech


In [10]:
import torch
import librosa
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

audio_array, sampling_rate = librosa.load(audio_path, sr=16000)

processor = AutoProcessor.from_pretrained("openai/whisper-small")
model = AutoModelForSpeechSeq2Seq.from_pretrained("openai/whisper-small")

inputs = processor(
    audio_array,
    sampling_rate=sampling_rate,
    return_tensors="pt"
)

predicted_ids = model.generate(inputs["input_features"])
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print("Auto-detected language transcription:")
print(transcription)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Auto-detected language transcription:
 I have a dream that one day this nation will rise up and live out the true meaning of its creed.
